# 🎯 Module 14: Computer Vision with Deep Learning

> **Stage:** Deep Learning  
> **Level:** Intermediate → Job-Ready  
> **Module:** Computer Vision with Deep Learning  
> **Framework:** PyTorch  
> **Date:** 14 June 2026  
> **Previous:** 13 — GANs

---

**Computer Vision (CV)** is the field of AI that enables computers to understand and work with **images and video**.

This module ties everything together: the CNN building blocks from Module 07, transfer learning from Module 11, and now the **complete production pipeline** — from data collection to deployment.

We'll cover:

- The main CV tasks: classification, detection, segmentation
- Image preprocessing and augmentation
- Transfer learning for CV
- A complete training pipeline
- Evaluation metrics: accuracy, precision, recall, IoU, mAP
- Video understanding
- Common CV problems: imbalance, leakage, wrong preprocessing
- A full end-to-end classification project

## 📚 Learning Objectives

By the end of this notebook you should be able to:

- Explain what computer vision is
- Distinguish **classification**, **detection**, and **segmentation**
- Build a **preprocessing + augmentation** pipeline
- Apply **transfer learning** to a custom image dataset
- Implement a **full training pipeline** with best checkpointing
- Compute **accuracy, precision, recall, and confusion matrix**
- Explain **IoU** and **mAP**
- Diagnose **class imbalance**, **overfitting**, and **data leakage**
- Design a **video understanding** pipeline
- Reason about **deployment** for a CV system

---
## ⚙️ 0. Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, random_split

import torchvision
from torchvision import datasets, transforms, models
from torchvision.models import resnet18, ResNet18_Weights

import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import time, copy, os

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else
                      'mps'  if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')
print(f'torchvision version: {torchvision.__version__}')

---
## 🧠 1. What Is Computer Vision?

**Computer Vision (CV)** is the field of AI that enables computers to understand and work with images and video.

Deep learning made CV effective because neural networks can **automatically learn visual features**.

### Common applications

- Image classification
- Object detection
- Image segmentation
- Face recognition
- OCR (optical character recognition)
- Medical imaging
- Autonomous systems
- Quality inspection

### Typical workflow

```text
Image / Video
     ↓
Preprocessing
     ↓
Deep Learning Model
     ↓
Prediction
     ↓
Post-processing
     ↓
Application
```

---
## 🎯 2. The Main Computer Vision Tasks

| Task | Output | Example |
| --- | --- | --- |
| **Classification** | Class | Cat / Dog |
| **Object Detection** | Class + Bounding Box | Find cars |
| **Segmentation** | Pixel-level labels | Separate road / person |
| **Image Generation** | New image | Synthetic face |
| **OCR** | Text | Read document |

### Classification

Answers: *"What is in this image?"*

```text
Image → CNN → [Cat: 0.92, Dog: 0.08]
```

### Object Detection

Answers: *"What objects are present and where?"*

```text
Image
 ↓
Model
 ↓
[Car, box, confidence]
[Person, box, confidence]
```

### Segmentation

Assigns labels to **individual pixels**:

```text
Background → 0
Road       → 1
Car        → 2
Person     → 3
```

---
## 🔧 3. Image Preprocessing

Deep learning models usually require **consistent input**. Common operations:

- Resize
- Normalize
- Convert image format
- Crop
- Augment
- Convert to tensor

### Example (torchvision)

```python
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
```

⚠️ The exact preprocessing should **match the model's expected input**.
When using pretrained models, use the transforms provided by the weights enum (`weights.transforms()`).

---
## 🎨 4. Data Augmentation

Augmentation creates **varied training examples** from existing images:

```text
Original
   ↓
 ┌───────────────┐
 │ Random Crop   │
 │ Flip          │
 │ Rotation      │
 │ Color Change  │
 └───────────────┘
```

### Example

```python
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])
```

Augmentation improves generalization, **but it must make sense for the problem**.

- Horizontal flips → ✅ natural images
- Horizontal flips → ❌ text recognition (would flip characters!)
- Rotation → ✅ microscopic slides
- Rotation → ❌ digit recognition (`6` ↔ `9`)

---
## 🔄 5. Transfer Learning for CV

Training a large vision model from zero often requires **substantial data and computation**.

A common approach:

```text
Pretrained Model
       ↓
Reuse learned visual features
       ↓
Replace final classifier
       ↓
Train on your dataset
```

### Example

```python
from torchvision.models import resnet18, ResNet18_Weights
import torch.nn as nn

model = resnet18(weights=ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 5)   # 5 classes
```

The detailed transfer-learning strategies were covered in Module 11 — here the focus is how they fit into a complete CV workflow.

---
## 🏗️ 6. The Complete Training Pipeline

```text
Dataset
  ↓
Train / Validation / Test Split
  ↓
Preprocessing + Augmentation
  ↓
Pretrained CNN
  ↓
Training
  ↓
Validation
  ↓
Best Checkpoint
  ↓
Test Evaluation
  ↓
Inference
```

### ⚠️ Critical rules

- **Augment only the training set** — never validation/test
- **Save the best checkpoint** based on validation performance
- **Touch the test set only once**, at the very end

---
## 📊 7. Evaluation Metrics

### Accuracy

$$\text{Accuracy} = \frac{\text{Correct Predictions}}{\text{Total Predictions}}$$

Useful when classes are **reasonably balanced**.

### Precision

$$\text{Precision} = \frac{TP}{TP + FP}$$

Of the samples predicted positive, how many were **actually positive**?

### Recall

$$\text{Recall} = \frac{TP}{TP + FN}$$

Of the **actual** positive samples, how many did the model **find**?

### IoU — Intersection over Union

Important for detection and segmentation:

$$\text{IoU} = \frac{\text{Area of Overlap}}{\text{Area of Union}}$$

```text
Predicted box ∩ Ground-truth box
---------------------------------
Predicted box ∪ Ground-truth box
```

### mAP — Mean Average Precision

Widely used for **object detection**. Considers both **classification confidence** and **localization quality** across predictions/classes.

---
## 🆚 8. Classification vs Detection vs Segmentation

| Task | Main Question | Output |
| --- | --- | --- |
| Classification | *What is it?* | Class |
| Detection | *What and where?* | Boxes + classes |
| Segmentation | *Which pixels belong to what?* | Pixel masks |

### Example: analyzing a road image

```text
Classification → "Road scene"

Detection → Car + Person + Traffic light

Segmentation → Road pixels + Car pixels + Person pixels
```

---
## 🎥 9. Video Understanding

Video is a **sequence of frames**:

```text
Frame 1 → Frame 2 → Frame 3 → ...
```

### A practical video pipeline

```text
Video
 ↓
Extract Frames
 ↓
Preprocess
 ↓
Vision Model
 ↓
Predictions
 ↓
Track / Aggregate Results
```

For more advanced temporal understanding, CNN features can be combined with **sequence models** (RNN/LSTM) or **Transformer-based** architectures.

---
## 🚨 10. Common CV Problems

### Class Imbalance

```text
Normal images:    9,500
Defective images:   500
```

Accuracy alone could hide **poor minority-class performance**.

**Solutions:**

- Better data collection
- Augmentation
- Class weighting
- Balanced sampling
- Use **F1, precision/recall** instead of just accuracy

### Data Leakage

Never allow **nearly identical images** from the same source to appear across training and test sets.

Example: photos of the same patient in both train and test (medical imaging).

### Wrong Preprocessing

Using preprocessing different from the pretrained model's expected pipeline can significantly reduce performance.

### Overfitting

Watch training and validation metrics:

```text
Training accuracy ↑
Validation accuracy → / ↓
```

Classic overfitting signature.

---
## 🛠️ 11. End-to-End Image Classification Project

We'll build a complete classifier for **CIFAR-10** (10 classes, 32×32 images) using **transfer learning** with a pretrained ResNet.

### Steps

1. **Dataset** — load CIFAR-10
2. **Split** — train / validation / test
3. **Augment** — training only
4. **Model** — pretrained ResNet-18
5. **Head** — replace final layer for 10 classes
6. **Train** — with best-checkpoint saving
7. **Evaluate** — accuracy, precision, recall, confusion matrix
8. **Inference** — on unseen images

---
## 📦 12. Loading & Preparing Data

In [ ]:
# Pretrained-model preprocessing (ImageNet stats)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

train_tf = transforms.Compose([
    transforms.Resize(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(224, padding=16),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_tf = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Load CIFAR-10 — subsample for speed
full_train = datasets.CIFAR10(root='./data', train=True,  download=True, transform=train_tf)
test_ds    = datasets.CIFAR10(root='./data', train=False, download=True, transform=eval_tf)

torch.manual_seed(42)
idx = torch.randperm(len(full_train))[:8000].tolist()
subset = Subset(full_train, idx)

n_val = 1500
n_tr  = len(subset) - n_val
train_ds, val_ds = random_split(subset, [n_tr, n_val],
                                generator=torch.Generator().manual_seed(42))

BATCH = 64
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=2)

CLASSES = full_train.classes
print('Train:', len(train_ds), ' Val:', len(val_ds), ' Test:', len(test_ds))
print('Classes:', CLASSES)

# Check class balance in training subset
train_labels = [full_train[i][1] for i in idx[:n_tr]]
print('Train label counts:', dict(Counter(train_labels)))

In [ ]:
# Visualize a batch
def denorm(img):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (img * std + mean).clamp(0, 1)

imgs, labels = next(iter(train_loader))
fig, axes = plt.subplots(1, 8, figsize=(16, 2.5))
for ax, img, lbl in zip(axes, imgs[:8], labels[:8]):
    ax.imshow(denorm(img).permute(1, 2, 0))
    ax.set_title(CLASSES[lbl], fontsize=9)
    ax.axis('off')
plt.tight_layout(); plt.show()

---
## 🧠 13. Building the Model

We use a pretrained ResNet-18 with a **fresh classifier head** for 10 classes.

The backbone is frozen (feature extraction). Fine-tuning is a natural follow-up exercise.

In [ ]:
torch.manual_seed(42)
model = resnet18(weights=ResNet18_Weights.DEFAULT)

# Freeze backbone
for p in model.parameters():
    p.requires_grad = False

# Replace head
model.fc = nn.Linear(model.fc.in_features, len(CLASSES))
model = model.to(device)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params    : {total:,}')
print(f'Trainable params: {trainable:,}   ({100*trainable/total:.2f}%)')

---
## 🏋️ 14. Training with Best Checkpointing

We track the best validation accuracy and **save a copy of the model state** whenever we improve.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        correct    += (logits.argmax(1) == yb).sum().item()
        total      += xb.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = criterion(logits, yb)
        total_loss += loss.item() * xb.size(0)
        correct    += (logits.argmax(1) == yb).sum().item()
        total      += xb.size(0)
    return total_loss / total, correct / total

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.fc.parameters(), lr=1e-3, weight_decay=1e-4)

EPOCHS = 6
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0.0
best_state   = None
t0 = time.time()

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    va_loss, va_acc = evaluate(model, val_loader, criterion)
    history['train_loss'].append(tr_loss); history['val_loss'].append(va_loss)
    history['train_acc'].append(tr_acc);   history['val_acc'].append(va_acc)

    marker = ''
    if va_acc > best_val_acc:
        best_val_acc = va_acc
        best_state = copy.deepcopy(model.state_dict())
        marker = '  ← best'

    print(f'Epoch {epoch:2d}/{EPOCHS} | '
          f'train_loss={tr_loss:.4f}  train_acc={tr_acc:.4f} | '
          f'val_loss={va_loss:.4f}  val_acc={va_acc:.4f}{marker}')

print(f'\nBest val acc: {best_val_acc:.4f} | total time: {time.time() - t0:.1f}s')
model.load_state_dict(best_state)   # restore best

In [ ]:
# Learning curves
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(history['train_loss'], label='train')
axes[0].plot(history['val_loss'],   label='val')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(history['train_acc'], label='train')
axes[1].plot(history['val_acc'],   label='val')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## 📊 15. Full Evaluation — Accuracy, Precision, Recall, Confusion Matrix

In [ ]:
@torch.no_grad()
def get_predictions(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    for xb, yb in loader:
        preds = model(xb.to(device)).argmax(1).cpu()
        all_preds.append(preds)
        all_labels.append(yb)
    return torch.cat(all_preds).numpy(), torch.cat(all_labels).numpy()

y_pred, y_true = get_predictions(model, test_loader)
acc = (y_pred == y_true).mean()
print(f'Test accuracy: {acc:.4f}')

# Confusion matrix (implemented with numpy, no sklearn needed)
C = np.zeros((len(CLASSES), len(CLASSES)), dtype=int)
for t, p in zip(y_true, y_pred):
    C[t, p] += 1

# Per-class precision & recall
print(f'\n{"Class":<10} {"Precision":>10} {"Recall":>10} {"Support":>10}')
print('-' * 44)
for i, cls in enumerate(CLASSES):
    tp = C[i, i]
    fp = C[:, i].sum() - tp
    fn = C[i, :].sum() - tp
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    support   = C[i, :].sum()
    print(f'{cls:<10} {precision:>10.3f} {recall:>10.3f} {support:>10}')

macro_p = np.mean([C[i,i] / max(C[:,i].sum(), 1) for i in range(len(CLASSES))])
macro_r = np.mean([C[i,i] / max(C[i,:].sum(), 1) for i in range(len(CLASSES))])
macro_f1 = 2 * macro_p * macro_r / max(macro_p + macro_r, 1e-9)
print(f'\nMacro precision: {macro_p:.4f}')
print(f'Macro recall   : {macro_r:.4f}')
print(f'Macro F1       : {macro_f1:.4f}')

In [ ]:
# Visualize the confusion matrix
plt.figure(figsize=(8, 7))
plt.imshow(C, cmap='Blues')
plt.colorbar(label='count')
plt.xticks(range(len(CLASSES)), CLASSES, rotation=45, ha='right')
plt.yticks(range(len(CLASSES)), CLASSES)
plt.xlabel('Predicted'); plt.ylabel('True')
plt.title(f'Confusion matrix — accuracy {acc:.3f}')

for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        plt.text(j, i, C[i, j], ha='center', va='center',
                 color='white' if C[i, j] > C.max() / 2 else 'black', fontsize=7)

plt.tight_layout(); plt.show()

---
## 🔍 16. Inference on Unseen Images

In [ ]:
# Predict a fresh batch and visualize with color-coded correctness
xb, yb = next(iter(test_loader))
xb, yb = xb.to(device), yb.to(device)
model.eval()
with torch.no_grad():
    preds = model(xb).argmax(1)

fig, axes = plt.subplots(2, 8, figsize=(16, 5))
for i, ax in enumerate(axes.flat):
    img = xb[i].cpu()
    ax.imshow(denorm(img).permute(1, 2, 0))
    p, t = preds[i].item(), yb[i].item()
    color = 'green' if p == t else 'red'
    ax.set_title(f'P: {CLASSES[p]}\nT: {CLASSES[t]}', fontsize=8, color=color)
    ax.axis('off')
plt.suptitle('Green = correct   |   Red = wrong')
plt.tight_layout(); plt.show()

---
## 📐 17. IoU Demo (Detection / Segmentation)

IoU measures how well two boxes overlap:

$$\text{IoU} = \frac{\text{Overlap Area}}{\text{Union Area}}$$

In [ ]:
def iou(boxA, boxB):
    # boxes are [x1, y1, x2, y2]
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    areaB = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    union = areaA + areaB - inter
    return inter / union if union > 0 else 0.0

gt  = [20, 20, 80, 80]
examples = {
    'Perfect match': [20, 20, 80, 80],
    'Slight shift':  [25, 25, 85, 85],
    'Partial':       [40, 40, 100, 100],
    'No overlap':    [200, 200, 260, 260],
}
for name, box in examples.items():
    print(f'{name:<15} IoU = {iou(gt, box):.3f}')

# Visualize the boxes
import matplotlib.patches as patches
fig, ax = plt.subplots(figsize=(5, 5))
ax.add_patch(patches.Rectangle((gt[0], gt[1]), gt[2]-gt[0], gt[3]-gt[1],
                               linewidth=2, edgecolor='green', facecolor='none', label='ground truth'))
for name, box in list(examples.items())[1:3]:
    ax.add_patch(patches.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1],
                                   linewidth=2, edgecolor='red', facecolor='none', label=name))
ax.set_xlim(0, 150); ax.set_ylim(150, 0)
ax.set_title('IoU between ground truth and predictions')
ax.legend(loc='upper right', fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## 🚨 18. Common CV Mistakes

### ❌ 1. Using accuracy on imbalanced data
A model that predicts *"normal"* for every sample gets 95% accuracy on 95/5 data — but is useless.

### ❌ 2. Augmenting validation/test
Augmentation is a **training-only** operation.

### ❌ 3. Data leakage
Nearly identical images from the same source split across train/test.

### ❌ 4. Wrong preprocessing
Not matching the pretrained model's expected mean/std/resize.

### ❌ 5. Tuning on the test set
Use **validation** for decisions; touch test once.

### ❌ 6. Forgetting `model.eval()` at inference
Dropout and BatchNorm behave differently during evaluation.

### ❌ 7. Confusing IoU with accuracy
IoU is for **boxes/masks**, not for classification.

---
## 🏋️ 19. Hands-On Exercises

### Exercise 1 — Fine-Tune the Backbone
Unfreeze `layer4` and use a smaller LR for it (see Module 11). Does accuracy improve?

### Exercise 2 — Handle Class Imbalance
Artificially make one class 10% as frequent. Retrain with:
- Class weights in the loss
- Weighted sampler
Report accuracy **and** F1.

### Exercise 3 — Different Backbone
Try `resnet50` or `efficientnet_b0`. Compare accuracy and training time.

### Exercise 4 — Build a Detection Demo
Use a pretrained torchvision detector (e.g. `fasterrcnn_resnet50_fpn`). Run it on a batch of images and visualize the boxes.

### Exercise 5 — Segmentation Demo
Use `torchvision.models.segmentation.deeplabv3_resnet50` on a single image and visualize the mask.

### Exercise 6 — Video Pipeline
Simulate frames by sliding a window over a batch of images. Classify each frame and aggregate the predictions.

### Exercise 7 — Deploy a Model
Save the model with `torch.save`, reload it, and run inference on a single image file.
```python
torch.save(model.state_dict(), 'cv_model.pt')
model.load_state_dict(torch.load('cv_model.pt', map_location=device))
```

---
## 🎤 20. Interview Questions

1. **What is computer vision?**  
   The field of AI that enables computers to understand images and video.

2. **Classification vs detection vs segmentation?**  
   Classification → one label per image.  
   Detection → labels + bounding boxes.  
   Segmentation → label per pixel.

3. **What is data augmentation?**  
   Creating varied training examples by applying label-preserving transformations.

4. **Why use transfer learning?**  
   To reuse features from large pretrained models — saving data, time, and compute.

5. **What is IoU?**  
   Intersection over Union — measures overlap between predicted and ground-truth boxes/masks.

6. **What is mAP?**  
   Mean Average Precision — a standard metric for detection that combines precision across classes.

7. **Why isn't accuracy always sufficient?**  
   On imbalanced data, high accuracy can hide poor minority-class performance.

8. **What causes overfitting in CV?**  
   Small datasets, too-large models, too many epochs, insufficient augmentation.

9. **What is data leakage?**  
   When test information leaks into training — e.g. near-duplicate images across splits.

10. **How would you deploy an image classification model?**  
    Export the weights, wrap in an inference function with correct preprocessing, and serve via API / edge / batch job.

---
## ✅ 21. Knowledge Check

- [ ] Explain what computer vision is
- [ ] Distinguish classification / detection / segmentation
- [ ] Build a preprocessing pipeline
- [ ] Apply data augmentation correctly
- [ ] Use a pretrained model for CV
- [ ] Build a full training pipeline with best checkpointing
- [ ] Compute accuracy, precision, recall, F1
- [ ] Read a confusion matrix
- [ ] Compute IoU
- [ ] Explain mAP
- [ ] Handle class imbalance
- [ ] Detect and prevent data leakage
- [ ] Reason about deployment

---
## 🏁 22. Key Takeaways

### The three main tasks

```text
Classification → What is it?
Detection      → What and where?
Segmentation   → Which pixels?
```

### The full pipeline

```text
Data → Preprocessing → Model → Evaluation → Inference → Deployment
```

### The most important practical rules

> **Augment training only.**  
> **Match the pretrained preprocessing.**  
> **Use accuracy + precision + recall + confusion matrix.**  
> **Touch the test set only once.**  
> **Deployment is not just the model — it's the entire pipeline.**

### The golden rule

> **A production CV system is *much more* than a neural network: data → preprocessing → model → evaluation → inference → deployment.**

---

### 🎓 You've completed the Deep Learning Deep Learning track!

You started with **single neurons** and worked your way through **CNNs, RNNs, LSTMs, Transformers, transfer learning, autoencoders, GANs, and the full CV pipeline**.

The **next step** is to build something of your own:

- Pick a real dataset
- Choose the right architecture for the task
- Apply the pipeline you learned here
- Evaluate carefully, then deploy

Deep learning is learned by **building**. Go build. 🚀